# 11주차 실습: 레이크하우스 통합과 품질 검증

강의: [PDF](../pdf/week11_data_pipeline_practice.pdf) · **PDF 실제 페이지 4–14, 18–30** (인쇄 번호와 다를 수 있음)

## 학습 목표
Landing→Gold 실행, 품질 격리, 재실행 멱등성과 기록을 검증한다.

권장 구성: 개념 확인 10분 → 코드 실행 30분 → 변경 실험 30분 → 결과 토의 20분.
설치는 README.md를 따릅니다. 새 커널에서 위에서 아래로 실행하세요.

In [ ]:
from pathlib import Path
import sys
here = Path.cwd().resolve()
root = next((p for p in [here, *here.parents] if (p / 'labkit').is_dir()), None)
if root is None:
    candidate = here / '강의자료' / '실습코드'
    if candidate.is_dir(): root = candidate
if root is None:
    raise RuntimeError('실습코드 폴더에서 Jupyter를 시작하세요. README.md를 확인하세요.')
sys.path.insert(0, str(root))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from labkit.common import ensure_data, DATA, output_dir, save_json
ensure_data()

## 단계 실행과 멱등성

In [ ]:
from labkit.lakehouse import run_pipeline, clean_events
out=output_dir('week11')
display(pd.DataFrame(run_pipeline(out)))
gold=pd.read_parquet(out/'gold.parquet')
run_pipeline(out)
pd.testing.assert_frame_equal(gold,pd.read_parquet(out/'gold.parquet'))
display(gold)
assert int(gold.events.sum())==2400

## 오류 주입과 격리

In [ ]:
raw=pd.read_json(DATA/'events.jsonl',lines=True)
dirty=pd.concat([raw,raw.iloc[:5]],ignore_index=True)
dirty.loc[0,'price']=-1
dirty['event_time']=dirty.event_time.astype(str)
dirty.loc[1,'event_time']='invalid'
clean,rejected=clean_events(dirty)
assert len(rejected)==2
assert clean.event_id.is_unique
assert len(clean)+len(rejected)+dirty.loc[~dirty.index.isin(rejected.index)].event_id.duplicated().sum()==len(dirty)
save_json(out/'injected_quality.json',{'input':len(dirty),'clean':len(clean),'rejected':len(rejected)})
print('격리:',len(rejected),'정제:',len(clean))

## 확장 과제 및 제출
scripts의 네 단계 파일을 순서대로 실행하고 실행 로그·품질 리포트·계층별 입출력을 제출하세요. 7주차 스트리밍과 batch 정제 조건을 동일하게 맞추고 같은 입력의 결과를 비교하세요. CDC/Delta MERGE는 이 로컬 스냅샷 구현의 확장 과제입니다.

제출: 실행한 노트북, 결과 표/그림, 변경한 조건과 해석. outputs/weekXX에 저장된 자료를 첨부하세요.

평가 기준: 개념 연결 25%, 실행·재현성 30%, 비교 실험 25%, 해석·한계 20%.